## Feed-Forward Neural Network from Stratch

### Libraries

In [103]:
import pandas as pd
import numpy as np

# interpretation of result
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# reference implementation
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import torch.optim as optim

### Load Case 1

In [104]:
df_train1 = pd.read_csv('../data/case1/train1.csv')
df_test1 = pd.read_csv('../data/case1/test1.csv')

### Change relevant columns

In [105]:
# speed in krpm, n
df_train1['krpm'] = df_train1['speed_rpm'] / 1000

# pressure rise across pump
df_train1['delta_p'] = (
    df_train1['outlet_pressure_bar'] - df_train1['inlet_pressure_bar']
)

# speed in krpm, n
df_test1['krpm'] = df_test1['speed_rpm'] / 1000

# pressure rise across pump
df_test1['delta_p'] = (
    df_test1['outlet_pressure_bar'] - df_test1['inlet_pressure_bar']
)


In [106]:
exclude_cols = [
    'pump_id',
    'elapsed_s',
    'operating_state',
    'inlet_pressure_bar',
    'outlet_pressure_bar',
    'flow_m3_h',
    'speed_rpm'
]

X_train = df_train1.drop(columns = exclude_cols)
y_train = df_train1['flow_m3_h']

X_test = df_test1.drop(columns = exclude_cols)
y_test = df_test1['flow_m3_h']

In [107]:
X_train

print(f"Max value in train set: {y_train.max()} \nMin value in train set: {y_train.min()}")

Max value in train set: 56.82320926887995 
Min value in train set: 0.354386134566899


In [108]:
X_train = np.array(X_train)
y_train = np.array(y_train)

X_test = np.array(X_test)
y_test = np.array(y_test)

### Create Helper Functions

In [109]:
class ReLU:
    ''' 
    ReLU (Rectified Linear Unit) activation
    '''

    @staticmethod
    def forward(inputs):
        ''' 
        Applies ReLU elementwise, returning max(0, x)
        ''' 
        return np.maximum(0, inputs, dtype=np.float64)

    @staticmethod
    def backward(inputs):
        ''' 
        Returns the ReLU derivative elementwise (1.0 where x > 0, else 0.0)

        Args:
            inputs (np.ndarray): Pre-activation values (same input to `forward`, not its output)
        '''
        return np.array(inputs > 0, dtype=np.float64)

class Cost:
    ''' 
    MSE cost and its gradient
    ''' 

    @staticmethod
    def cost_forward(ypred, ytrue):
        ''' 
        Returns the mean squared error over all elements
        ''' 
        return np.mean((ytrue - ypred) ** 2)

    @staticmethod
    def cost_backward(ypred, ytrue):
        ''' 
        Returns d(cost)/d(ypred), divided by the batch size (axis 0)
        ''' 
        return 2 * (ypred - ytrue) / ytrue.shape[0]

class SGD:
    ''' 
    Plain SGD - Stochastic Gradient Descent

    All input arrays are of shape (features, batch_size)
    '''

    def __init__(self, learning_rate=0.005):
        '''
        Initalizing learning rate
        ''' 
        self.lr = learning_rate

    def generate_nabla(self, delta, prev_activation):
        ''' 
        Computes scaled weight and bias for one layer

        Args:
            delta (np.ndarray): Error term for layer
            prev_activation (np.ndarray): Previous layer's activations

        Returns:
            nabla_w: Weight update, scaled by learning rate
            nabla_b: Bias update, scaled by learning rate
        ''' 
        nabla_w = self.lr * (delta @ prev_activation.T)
        nabla_b = self.lr * delta.sum(axis = 1, keepdims = True)
        return nabla_w, nabla_b

### Feed-Foward Neural Network from Scratch

In [110]:
class FNN:
    ''' 
    Fully connected layer

    Arrays are of shape (features, batch_size)

    Bias and weights are None until initialized
    '''
    
    def __init__(self, units: int, activation=None, input_dim: int = False):
        ''' 
        Initializes the layer configuration.

        Args:
            units (int): Number of neurons in this layer.
            activation (object / class): Activation object with `forward` and `backward`
                method. If None, the layer is linear.
            input_dim (int): Number of input features.
        ''' 
        self.units = units
        self.activation = activation
        self.input_dim = input_dim

        self.weights = None
        self.bias = None

    def initialize(self, prev_units: int): 
        ''' 
        Initializes weights with He initialization and biases with zeros.

        Args:
            prev_units (int): Number of units in the previous layer (or input
                features if first layer).
        '''
        self.weights = np.random.normal(0 , np.sqrt(2 / prev_units) , (self.units, prev_units)) # He initalization
        self.bias = np.zeros((self.units , 1)) # Match PyTorch implementation better

    def forward(self, inputs):
        ''' 
        Computes the layer's output for a batch.

        Args:
            inputs (np.ndarray): Previous layer's activations, shape (prev_units, batch_size).

        Returns:
            z: Pre-activation values, shape (units, batch_size)
            a: Post-activation values, shape (units, batch_size)
        ''' 
        z = self.weights @ inputs + self.bias
        if self.activation is not None:
            return z, self.activation.forward(z)
        return z, z

    def backward(self, propagated_error, current_z): 
        ''' 
        Backpropagates error through layer.

        Args:
            propagated_error (np.ndarray): Gradient of cost w.r.t this layer's output, shape (units, batch_size)
            current_z (np.ndarray): Pre-activation values saved from `forward`, shape (units, batch_size)

        Returns:
            delta: Gradient of cost w.r.t z, shape (units, batch_size)
            next_propagated_error: Gradient of cost w.r.t previous layer's output, shape (prev_units, batch_size)
        '''
        delta = propagated_error
        if self.activation is not None:
            delta = delta * self.activation.backward(current_z)
        next_propagated_error = self.weights.T @ delta
        return delta, next_propagated_error

    def update(self, nabla_w, nabla_b):
        ''' 
        Applies a gradient descent step to the weights and biases

        Args:
            nabla_w (np.ndarray): Weight update, shape (units, prev_units), scaled by learning rate
            nabla_b (np.ndarray): Bias update, shape (units, 1), scaled by learning rate
        ''' 
        self.weights = self.weights - nabla_w
        self.bias = self.bias - nabla_b

In [111]:
import numpy as np
import copy

class Sequential:
    '''
    Linear stack of layers trained with backpropagation.

    Input arrays shaped as (n_samples, n_features) and transposes to (features, batch_size)
    ''' 

    def __init__(self, layers = None):
        ''' 
        Initializes the model.
        '''
        self.layers = layers if layers is not None else []


    def add(self, layer):
        ''' 
        Appends a layer to the end of the model.
        '''
        self.layers.append(layer)


    def _initialize_weights_and_bias(self, input_shape):
        ''' Initializes every layer's parameters

        Args:
            input_shape: Number of input features used in first layer.
        ''' 
        for layer in self.layers:
            layer.initialize(input_shape)
            input_shape = layer.units


    def compile(self, optimizer=None, cost=Cost):
        ''' 
        Makes optimizer and cost function.

        Args:
            optimizer: Optimizer exposing `generate_nabla`. Defaults to
                `SGD()` with its default learning rate
            cost: Cost class exposing `cost_forward` and `cost_backward`.
        ''' 
        self.optimizer = optimizer if optimizer is not None else SGD()
        self.cost = cost
    

    def evaluate(self, X, y):
        ''' 
        Computes predictions and the cost on a dataset.

        Args:
            X (np.ndarray): Input data
            y (np.ndarray): Target values. Flattened to 1-D

        Returns:
                ypred: Predictions, shape (n_out, n_samples).
                cost: Scalar cost of `ypred` against `y`.
        '''
        y = y.reshape(-1)
        ypred = self.predict(X)
        cost = self.cost.cost_forward(ypred, y)

        return ypred , cost


    def predict(self, X):
        '''
        Runs a forward pass through all layers.

        Args:
            X (np.ndarray): Input data

        Returns:
            Output activations of the last layer, shape (n_out, n_samples).
        '''
        _, a = self.layers[0].forward(X.T)
        for j in range(1, len(self.layers)):
            _, a = self.layers[j].forward(a)
        return a


    def fit(self, X, y, batch_size=1, epochs=1, verbose=1, save_best=False):
        ''' 
        Trains the model with mini-batch gradient descent.

        Weights are initialized on the first call if they don't exist yet,
        so calling `fit` again continues training from the current weights.
        Parameters are updated layer by layer during the backward pass.

        Args:
            X (np.ndarray): Training inputs
            y (np.ndarray): Training targets, flattened to shape (n_samples,)
            batch_size: Number of samples per gradient update.
            epochs: Number of passes over the training set.
            verbose: If 1, prints the cost after every epoch.
            save_best: If True, restores the layers from the epoch with the
                lowest cost once training finishes.

        Returns:
            List with one cost value per epoch, measured on the full
            training set after that epoch.

        Raises:
            ValueError: If the first layer has no `input_dim`.
        ''' 
        y = y.reshape(-1)
        if not self.layers[0].input_dim:
            raise ValueError("Please specify input_dim on the first layer")

        number_of_layers = len(self.layers)
        if self.layers[0].weights is None:  self._initialize_weights_and_bias(input_shape=self.layers[0].input_dim)

        costs = []
        X = np.asarray(X, dtype=np.float64)
        y = np.asarray(y, dtype=np.float64).reshape(-1)
        total_samples = len(y)

        best_layers = None
        best_cost = np.inf

        for i in range(epochs):
            idx = np.random.permutation(total_samples)
            for start in range(0, total_samples, batch_size):
                b = idx[start:start + batch_size]
                X_taken, y_taken = X[b].T, y[b]

                # Forward pass
                z, a = self.layers[0].forward(X_taken)
                zs, activations = [z], [a]
                for j in range(1, number_of_layers):
                    z, a = self.layers[j].forward(activations[-1])
                    zs.append(z)
                    activations.append(a)

                # Backward pass
                propagated_error = self.cost.cost_backward(activations[-1], y_taken)
                for j in range(number_of_layers - 1, -1, -1):
                    delta, propagated_error = self.layers[j].backward(
                        propagated_error, zs[j]
                    )

                    prev_activation = X_taken if j == 0 else activations[j - 1]

                    nabla_w, nabla_b = self.optimizer.generate_nabla(delta, prev_activation)
                    self.layers[j].update(nabla_w, nabla_b)
                    
             # -------------------------------
            cost = self.cost.cost_forward(self.predict(X), y)

            # inserted to check RunTimeWarning 
            if not np.isfinite(cost):
                print(f"Training diverged at epoch {i+1}")
                return None

            if cost < best_cost:
                best_cost = cost
                best_layers = copy.deepcopy(self.layers)

            costs.append(cost)

            if verbose == 1:
                print(f"Epoch ({i + 1}/{epochs}), cost = {cost}")
            # -------------------------------
        
        if save_best and best_layers is not None:            
            self.layers = best_layers
            print("Best cost:", best_cost)

        return costs

### Split dataset

In [112]:
new_X_train, X_val, new_y_train, y_val = train_test_split(
    X_train, y_train, test_size = 0.2, random_state = 42
)

x_scaler = StandardScaler()
new_X_train = x_scaler.fit_transform(new_X_train)
X_val = x_scaler.transform(X_val)
X_test = x_scaler.transform(X_test)

y_scaler = StandardScaler()
new_y_train = y_scaler.fit_transform(new_y_train.reshape(-1 , 1))
y_val = y_scaler.transform(y_val.reshape(-1 , 1))
y_test = y_scaler.transform(y_test.reshape(-1 , 1))

# full train1.csv – don't use until final model
X_train = x_scaler.transform(X_train)
y_train = y_scaler.transform(y_train.reshape(-1 , 1))

### Tune hyperparameters for Scratch model

In [125]:
np.random.seed(42)

def score(X_s, y_s):
    pred = y_scaler.inverse_transform(model.predict(X_s).reshape(-1, 1)).ravel()
    true = y_scaler.inverse_transform(np.asarray(y_s).reshape(-1, 1)).ravel()
    return mean_squared_error(true, pred), r2_score(true, pred)

best_val_mse = float("inf")
best_val_r2 = -float("inf")
best_neuron1 = best_neuron2 = best_lr  = best_batch = best_epoch =  None

for neuron1 in [16 , 32 , 64 , 128]:
   for neuron2 in [16 , 32 , 64 , 128 , 160]:
       for lr in [0.001, 0.003, 0.01, 0.02, 0.03]:
           for batch in [8 , 16 , 32 , 64]:
               for epoch in [50]:
                    np.random.seed(42)
                    model = Sequential()
                    model.add(FNN(neuron1, ReLU, input_dim=new_X_train.shape[1]))
                    model.add(FNN(neuron2, ReLU))
                    model.add(FNN(1))
                    model.compile(optimizer=SGD(lr), cost=Cost)

                    costs = model.fit(new_X_train, new_y_train, batch_size=batch, epochs=epoch, verbose=0)


                    if costs is None:
                        print(
                            f"Skipping unstable config: "
                            f"n1={neuron1}, "
                            f"n2={neuron2}, "
                            f"lr={lr}, "
                            f"batch={batch}, "
                            f"epoch={epoch}"
                        )
                        continue
                    
                    # train_mse , train_R2  = score(new_X_train, new_y_train)
                    val_mse , val_r2 = score(X_val, y_val)

                    if val_mse <  best_val_mse:
                        best_val_mse = val_mse
                        best_val_r2 = val_r2 
                        best_neuron1 = neuron1
                        best_neuron2 = neuron2
                        best_lr = lr
                        best_batch = batch
                        best_epoch = epoch
        

print(
    f"Best neuron 1 = {best_neuron1}, "
    f"Best neuron 2 = {best_neuron2}, "
    f"Best learning rate = {best_lr}, "
    f"Best batch size = {best_batch}, "
    f"Best epoch = {best_epoch}: "
    f"Best validation MSE = {best_val_mse:.3f}, "
    f"Best validation R2 = {best_val_r2:.3f}"
)

Best neuron 1 = 32, Best neuron 2 = 160, Best learning rate = 0.03, Best batch size = 8, Best epoch = 50: Best validation MSE = 1.163, Best validation R2 = 0.992


### Implementing reference model

##### Preparing data

In [114]:
X_train_tensor = torch.FloatTensor(new_X_train) 
y_train_tensor = torch.FloatTensor(new_y_train).reshape(-1 , 1)

X_val_tensor = torch.FloatTensor(X_val)
y_val_tensor = torch.FloatTensor(y_val).reshape(-1 , 1)

X_test_tensor = torch.FloatTensor(X_test)
y_test_tensor = torch.FloatTensor(y_test).reshape(-1 , 1)

# Create datasets
train_dataset = TensorDataset(X_train_tensor, y_train_tensor) 
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=batch, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch, shuffle=False)

##### Best parameters referenced from manual gridsearch

In [115]:
n1 , n2 , lr , batch , num_epoch = best_neuron1 , best_neuron2 , best_lr ,  best_batch , best_epoch
print(n1 , n2 , lr , batch , num_epoch)

32 160 0.03 8 50


In [116]:
# Model
class Regressor(nn.Module):
    def __init__(self, n_in, n1, n2):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(n_in, n1),
            nn.ReLU(),

            nn.Linear(n1, n2),
            nn.ReLU(),

            nn.Linear(n2, 1)        # linear output
        )

    def forward(self, x):
        return self.network(x)

torch.manual_seed(42)
ref_model = Regressor(X_train_tensor.shape[1], n1, n2)
print(ref_model)

# Loss and optimizer
criterion = nn.MSELoss()
optimizer = optim.SGD(ref_model.parameters(), lr=lr)

# Training loop
for epoch in range(num_epoch):
    ref_model.train()
    running_loss = 0.0

    for inputs, targets in train_loader:
        optimizer.zero_grad()
        outputs = ref_model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{num_epoch}], Loss: {avg_loss:.4f}")

# Evaluation on validation set, in original units
ref_model.eval()
all_predictions = []
all_targets = []

with torch.no_grad():
    for inputs, targets in val_loader:
        outputs = ref_model(inputs)
        all_predictions.append(outputs.numpy())
        all_targets.append(targets.numpy())

pred = y_scaler.inverse_transform(np.concatenate(all_predictions).reshape(-1, 1)).ravel()
true = y_scaler.inverse_transform(np.concatenate(all_targets).reshape(-1, 1)).ravel()

print(f"Validation MSE: {mean_squared_error(true, pred):.3f}")
print(f"Validation R2:  {r2_score(true, pred):.3f}")

Regressor(
  (network): Sequential(
    (0): Linear(in_features=3, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_features=160, bias=True)
    (3): ReLU()
    (4): Linear(in_features=160, out_features=1, bias=True)
  )
)
Epoch [1/50], Loss: 0.9145
Epoch [2/50], Loss: 0.7564
Epoch [3/50], Loss: 0.6389
Epoch [4/50], Loss: 0.5323
Epoch [5/50], Loss: 0.4331
Epoch [6/50], Loss: 0.3417
Epoch [7/50], Loss: 0.2630
Epoch [8/50], Loss: 0.2043
Epoch [9/50], Loss: 0.1632
Epoch [10/50], Loss: 0.1363
Epoch [11/50], Loss: 0.1186
Epoch [12/50], Loss: 0.1065
Epoch [13/50], Loss: 0.0975
Epoch [14/50], Loss: 0.0889
Epoch [15/50], Loss: 0.0812
Epoch [16/50], Loss: 0.0735
Epoch [17/50], Loss: 0.0677
Epoch [18/50], Loss: 0.0611
Epoch [19/50], Loss: 0.0558
Epoch [20/50], Loss: 0.0511
Epoch [21/50], Loss: 0.0467
Epoch [22/50], Loss: 0.0430
Epoch [23/50], Loss: 0.0397
Epoch [24/50], Loss: 0.0364
Epoch [25/50], Loss: 0.0341
Epoch [26/50], Loss: 0.0319
Epoch [27/50], Loss: 0.0295

### Final model

In [117]:
model_s = Sequential()
model_s.add(FNN(n1, ReLU, input_dim=X_train.shape[1]))
model_s.add(FNN(n2, ReLU))
model_s.add(FNN(1))
model_s.compile(optimizer=SGD(lr), cost=Cost)

costs_s = model_s.fit(X_train, y_train, batch_size=batch, epochs=num_epoch, verbose=0)

pred = y_scaler.inverse_transform(model_s.predict(X_test).reshape(-1, 1)).ravel()
true = y_scaler.inverse_transform(np.asarray(y_test).reshape(-1, 1)).ravel()

print("Test MSE:", mean_squared_error(true, pred), " Test R2:", r2_score(true, pred))

Test MSE: 1.7868113758521749  Test R2: 0.9877403269822792


In [118]:
X_train_tensor = torch.FloatTensor(X_train)
y_train_tensor = torch.FloatTensor(y_train).reshape(-1, 1)
full_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=batch, shuffle=True)

torch.manual_seed(42)
ref_model = Regressor(X_train_tensor.shape[1], n1, n2)
criterion = nn.MSELoss()
optimizer = optim.SGD(ref_model.parameters(), lr=lr)

costs_ref = []
for epoch in range(num_epoch):
    ref_model.train()
    running_loss = 0.0
    for inputs, targets in full_loader:
        optimizer.zero_grad()
        loss = criterion(ref_model(inputs), targets)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    costs_ref.append(running_loss / len(train_loader))

ref_model.eval()
with torch.no_grad():
    pred_ref = ref_model(X_test_tensor).numpy().reshape(-1, 1)

predict_ref = y_scaler.inverse_transform(pred_ref).ravel()
true_ref = y_scaler.inverse_transform(y_test_tensor.numpy().reshape(-1, 1)).ravel()

test_mse_ref = mean_squared_error(true_ref, predict_ref)
test_r2_ref = r2_score(true_ref, predict_ref)

print(f"Test MSE for reference implementation = {test_mse_ref:.3f}, "
      f"Test R2 for reference implementation = {test_r2_ref:.3f}")

Test MSE for reference implementation = 1.015, Test R2 for reference implementation = 0.993


### Correctness

In [123]:
np.sqrt(np.mean((pred - predict_ref)**2))

np.float64(0.6565375268835495)